**DATA CLEANING NOTES PER DATASET**

1. Gemini Dataset
    - A bit easy to clean if we were to base the cleaning on the

2. Claude Dataset
    - Needs a bit of cleaning since it has a lot of curly braces
    - It has equations and code 
    - to clean: remaining greek operators, fractions, and code

3. MGTBench Dataset (ChatGPT)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

4. MGTBench Dataset (Human-written)
    - heavy in data cleaning especially for subjects in physics
    - has new lines that needs to be removed

5. BAWE Corpus Dataset
    - Needs the most cleaning since it an HTML file 
    - Need to convert it into an HTML file so we can clean it properly

In [24]:
# Import the necessary libraries for cleaning the data
import pandas as pd
from pathlib import Path

# Import all cleaning functions from the text_cleaning module
from utils.text_cleaning import (
    clean_claude_dataset,
    clean_pipeline,
    is_academic_content,
    contains_foreign_language,
    placeholder_density,
    placeholder_density_windowed,
    extract_claude_prompt_and_response
)

pd.set_option('display.max_colwidth', 150)

print("Libraries has been imported!")

Libraries has been imported!


In [25]:
# Establish the directories where the data will be read and stored after processing
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent.parent
DATA_DIR = PROJECT_ROOT / "data"

# Directories of the Raw Datasets
RAW_AI_DIR = DATA_DIR / "raw" / "ai"
RAW_HUMAN_DIR = DATA_DIR / "raw" / "human"

# Directories of the Processed Datasets where it will be stored after cleaning the data
PROCESSED_AI_DIR = DATA_DIR / "processed" / "ai"
PROCESSED_HUMAN_DIR = DATA_DIR / "processed" / "human"

# To ensure that the out processed directories exist
PROCESSED_AI_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_HUMAN_DIR.mkdir(parents=True, exist_ok=True)

print("Data Cleaning Paths Ready:")
print(f"  Reading AI Raw Data:        {RAW_AI_DIR.resolve()}")
print(f"  Reading Human Raw Data:     {RAW_HUMAN_DIR.resolve()}")
print(f"  Saving AI Processed Data:   {PROCESSED_AI_DIR.resolve()}")
print(f"  Saving Human Processed Data:{PROCESSED_HUMAN_DIR.resolve()}")


Data Cleaning Paths Ready:
  Reading AI Raw Data:        C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\ai
  Reading Human Raw Data:     C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\raw\human
  Saving AI Processed Data:   C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai
  Saving Human Processed Data:C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\human


**METHODS FOR CLEANING DATA**

- Cleans math texts
- Cleans texts that has citations
- Cleans texts that has code in it
- Strips reference lists
- Cleans texts that has any numberings in it
- Checks whether a text is creative works and it will remove that since creatives is not considered as an academic text

Since most datasets have a lot of placeholders which can render the dataset unsable/useful for both spaCy and ELECTRA. 

In [28]:
# Run Claude dataset cleaning
claude_dataset = RAW_AI_DIR / 'claude_dataset.csv'
df_cleaned_claude = clean_claude_dataset(claude_dataset, PROCESSED_AI_DIR, sample_size=None)

Loading all rows from claude_dataset.csv...
Cleaning & filtering Claude dataset...


Processing Rows: 100%|██████████| 9941/9941 [08:25<00:00, 19.68it/s]



--- CLEANING SUMMARY ---


,Metric,Count
0,Total Rows Loaded,9941
1,Dropped (Non-Academic/Creative),5820
2,Dropped (Foreign-Language Content),691
3,Dropped (Too Placeholder-Dense),1
4,Dropped (Locally Dense Cluster),112
5,Total Academic Rows Kept,3317
6,[[EQUATION]] Tags Inserted,11804
7,[[CODE]] Tags Inserted,5575
8,[[CITATION]] Tags Inserted,791
9,[[COMPLEXITY]] Tags Inserted,625



Successfully saved cleaned dataset (3317 rows) to:
  C:\Users\Alden Olmedo\Documents\VSCode\hybrid-ai-framework\data\processed\ai\claude_dataset_cleaned.csv

--- SAMPLE CLEANED DATA (FIRST 20 ROWS) ---


,prompt,cleaned_text
0,Create a function in Python code that accepts a string input and outputs the number of vowels in the string. The function signature and a sample c...,"Here's a Python function that counts the number of vowels in a given string: [[CODE]] This function does the following: We define a set of vowels,..."
1,"Write a well-articulated series of paragraphs explaining how to effectively structure one's writing through the use of topic sentences, transition...","To effectively structure one's writing, it's important to employ clear topic sentences, smooth transitions, a consistent theme, and proper punctua..."
2,Derive a proof for the Pythagorean Theorem relating the sides of a right triangle and use it to help students understand and remember the key math...,[[EQUATION]] I'll provide a proof for the Pythagorean Theorem and then discuss how to help students understand and remember this important mathema...
3,•Faraday's experiments on induction which established the basic principles. Discuss how Faraday's observations lead to the concept of magnetic fl...,"Faraday's Experiments and Magnetic Flux: In the 1830s, Michael Faraday conducted a series of experiments that established the fundamental principl..."
4,Imagine that you have two claims that appear to conflict with one another. Analyze the reasons and evidence provided for both claims. Look for way...,Thank you for this interesting thought exercise. I'll do my best to analyze and reconcile two hypothetical conflicting claims in a reasoned way. S...
5,Explain how to solve a system of three nonlinear equations in three unknowns using Newton's method. Provide detailed step-by-step instructions alo...,Newton's method can be used to solve systems of nonlinear equations by finding successive approximations to the solution. For a system of 3 equati...
6,Summarize the U.S court case Miranda v. Arizona (1966) in which the Supreme Court established that the police must inform individuals of their Mir...,Miranda [[CITATION]] was a landmark U. Supreme Court case that established important constitutional protections for criminal suspects during polic...
7,"Write a short Python program that generates 10 random integer numbers between 1 and 100, calculates the sum and average of those numbers, and pri...",Here's a short Python program that does what you've described: [[CODE]] This program does the following: It imports the [[CODE]] module to generat...
8,"Explain in detail how interest rates set by central banks impact economic growth and stability. Consider factors like investment, consumption, emp...","Interest rates set by central banks play a crucial role in shaping economic growth and stability. These rates, often referred to as the ""policy ra..."
9,"Write a step-by-step procedure that thoroughly explains how you would debug an error in a Python program, covering multiple possible causes of and...","Here's a comprehensive step-by-step procedure for debugging errors in a Python program, covering multiple error types, causes, solutions, and debu..."


In [30]:
def clean_mgtbench_human_dataset(dataset):

    pass

In [31]:
import re
import pandas as pd
import IPython.display as ipd


# Placeholder types used by the cleaning pipeline
PLACEHOLDER_NAMES = (
    "EQUATION",
    "CODE",
    "CITATION",
    "COMPLEXITY",
    "URL",
    "FOREIGN"
)

PLACEHOLDER_PATTERN = (
    r"\[\[(?:EQUATION|CODE|CITATION|COMPLEXITY|URL|FOREIGN)\]\]"
)


# Replace code blocks and inline code with a placeholder
def clean_code_texts(text):
    if not isinstance(text, str):
        return text
    text = re.sub(
        r"```[\s\S]*?```",
        " [[CODE]] ",
        text)

    text = re.sub(
        r"`[^`\n]+`",
        " [[CODE]] ",
        text)
    return text


# Replace URLs with a placeholder
def clean_url(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"https?://\S+|www\.\S+",
        " [[URL]] ",
        text,
        flags=re.IGNORECASE)
    return text


# Replace foreign-language scripts with a placeholder
def clean_foreign_script(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"[\u0400-\u04FF]+",
        " [[FOREIGN]] ",
        text)
    text = re.sub(
        r"[\u0600-\u06FF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\u4E00-\u9FFF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\u3040-\u30FF]+",
        " [[FOREIGN]] ",
        text
    )
    text = re.sub(
        r"[\uAC00-\uD7AF]+",
        " [[FOREIGN]] ",
        text
    )
    return text


# Replace common algorithm complexity notation
def clean_complexity_notation(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\bO\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )
    text = re.sub(
        r"\bΩ\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )
    text = re.sub(
        r"\bΘ\s*\([^)]{1,100}\)",
        " [[COMPLEXITY]] ",
        text
    )

    return text


# Replace common academic citation formats
def clean_citations(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\[\s*\d+(?:\s*[-,]\s*\d+)*\s*\]",
        " [[CITATION]] ",
        text
    )

    text = re.sub(
        r"\\(?:cite|citep|citet|citealp|citeauthor|citeyear)"
        r"(?:\[[^\]]*\])?"
        r"\{[^}]*\}",
        " [[CITATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\([A-Z][A-Za-z.\-]+"
        r"(?:\s+et al\.)?"
        r",?\s*\d{4}[a-z]?\)",
        " [[CITATION]] ",
        text
    )

    text = re.sub(
        r"\b[A-Z][A-Za-z.\-]+"
        r"(?:\s+et al\.)?"
        r"\s*\(\d{4}[a-z]?\)",
        " [[CITATION]] ",
        text
    )

    return text


# Detect large LaTeX equations and replace them with one placeholder
def collapse_complete_equations(text):
    if not isinstance(text, str):
        return text

    equation_pattern = re.compile(
        r"(?<!\w)"
        r"[\(\[]?\s*"
        r"(?:\\{1,2})"
        r"(?:"
        r"frac|"
        r"dfrac|"
        r"tfrac|"
        r"sqrt|"
        r"sum|"
        r"prod|"
        r"int|"
        r"oint|"
        r"lim|"
        r"mathcal|"
        r"mathrm|"
        r"mathbf|"
        r"mathit|"
        r"operatorname|"
        r"partial|"
        r"nabla"
        r")"
        r"[^.!?\n]{0,2000}",
        flags=re.IGNORECASE
    )

    def replace_equation(match):
        equation = match.group(0)

        has_equals = "=" in equation

        has_latex = bool(
            re.search(
                r"\\{1,2}[A-Za-z]+",
                equation
            )
        )

        has_math_structure = bool(
            re.search(
                r"[\^_{}]",
                equation
            )
        )

        if has_latex and (
            has_equals or has_math_structure
        ):
            return " [[EQUATION]] "

        return equation

    text = equation_pattern.sub(
        replace_equation,
        text
    )

    return text


# Clean LaTeX, math notation, and equation symbols
def clean_math_texts(text):
    if not isinstance(text, str):
        return text

    text = text.replace("\\n", " ")
    text = text.replace("\\t", " ")
    text = text.replace("\r", " ")
    text = text.replace("\n", " ")

    text = collapse_complete_equations(text)

    text = re.sub(
        r"\\begin\{[^}]+\}[\s\S]*?\\end\{[^}]+\}",
        " [[EQUATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\$\$[\s\S]*?\$\$",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\\[[\s\S]*?\\\]",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\\([\s\S]*?\\\)",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\$(?!\$)[^\$\n]+?\$(?!\$)",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\(?:"
        r"ref|eqref|pageref|label|"
        r"section|subsection|subsubsection|paragraph|"
        r"textbf|textit|texttt|text|"
        r"mathrm|mathbf|mathit|mathsf|mathtt|"
        r"operatorname|"
        r"frac|dfrac|tfrac|sqrt|"
        r"sum|prod|int|oint|lim|"
        r"partial|nabla"
        r")"
        r"\*?"
        r"(?:\s*\{[^{}]*\})+",
        " [[EQUATION]] ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\\[A-Za-z]+\*?(?:\{[^{}]*\})?",
        " [[EQUATION]] ",
        text
    )

    text = re.sub(
        r"\\[A-Za-z]+",
        " [[EQUATION]] ",
        text
    )

    math_symbols = (
        "∑∏∫∮√∞≈≠≤≥±×÷"
        "∂∇∆∈∉⊂⊃⊆⊇"
        "∀∃∄"
        "→←↔⇒⇐⇔"
    )

    text = re.sub(
        "[" + re.escape(math_symbols) + "]",
        " [[EQUATION]] ",
        text
    )

    return text


# Remove everything after a References/Bibliography section
def strip_reference_list(text):
    if not isinstance(text, str):
        return text

    match = re.search(
        r"(?:^|\n)\s*"
        r"(?:References|Bibliography|Sources)"
        r"\s*:?\s*",
        text,
        flags=re.IGNORECASE
    )

    if match:
        text = text[:match.start()]

    return text


# Remove list numbering while keeping the actual text
def clean_list_numbering(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"(?<!\w)\d{1,2}[.)]\s+(?=[A-Za-z])",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)[A-Za-z][.)]\s+(?=[A-Za-z])",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)"
        r"(?:i{1,3}|iv|v|vi{0,3}|ix|x)"
        r"[.)]\s+",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"[•●▪◦]\s*",
        " ",
        text
    )

    return text


# Remove leftover LaTeX characters and mathematical noise
def clean_residual_math_noise(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"\\[A-Za-z]+",
        " ",
        text
    )

    text = re.sub(
        r"[{}]",
        " ",
        text
    )

    text = re.sub(
        r"\\+",
        " ",
        text
    )

    text = text.replace(
        "~",
        " "
    )

    text = re.sub(
        r"(?<!\w)[_^]+(?!\w)",
        " ",
        text
    )

    text = re.sub(
        r"([.,;:!?])\1+",
        r"\1",
        text
    )

    return text


# Remove parentheses surrounding placeholders
def clean_placeholder_parentheses(text):
    if not isinstance(text, str):
        return text

    previous = None

    while previous != text:
        previous = text

        text = re.sub(
            rf"\(\s*({PLACEHOLDER_PATTERN})\s*\)",
            r"\1",
            text
        )

    return text


# Make all placeholder formats consistent
def normalize_placeholders(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"<EQUATION>",
        "[[EQUATION]]",
        text
    )

    text = re.sub(
        r"<CODE>",
        "[[CODE]]",
        text
    )

    text = re.sub(
        r"<CITATION>",
        "[[CITATION]]",
        text
    )

    text = re.sub(
        r"<COMPLEXITY>",
        "[[COMPLEXITY]]",
        text
    )

    text = re.sub(
        r"<URL>",
        "[[URL]]",
        text
    )

    text = re.sub(
        r"<FOREIGN>",
        "[[FOREIGN]]",
        text
    )

    text = re.sub(
        r"\[\[\s*EQUATION\s*\]\]",
        "[[EQUATION]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*CODE\s*\]\]",
        "[[CODE]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*CITATION\s*\]\]",
        "[[CITATION]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*COMPLEXITY\s*\]\]",
        "[[COMPLEXITY]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*URL\s*\]\]",
        "[[URL]]",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\[\[\s*FOREIGN\s*\]\]",
        "[[FOREIGN]]",
        text,
        flags=re.IGNORECASE
    )

    # Convert malformed equation brackets to one standard placeholder
    text = re.sub(
        r"\[{1,2}\s*EQUATION\s*\]{1,2}",
        "[[EQUATION]]",
        text,
        flags=re.IGNORECASE
    )

    return text


# Collapse repeated placeholders such as [[EQUATION]] [[EQUATION]]
def collapse_consecutive_placeholders(text):
    if not isinstance(text, str):
        return text

    pattern = rf"(?:{PLACEHOLDER_PATTERN}\s*){{2,}}"

    def replace_cluster(match):
        placeholders = re.findall(
            PLACEHOLDER_PATTERN,
            match.group(0)
        )

        if not placeholders:
            return ""

        return placeholders[0] + " "

    text = re.sub(
        pattern,
        replace_cluster,
        text
    )

    return text


# Collapse equation fragments that were split into multiple placeholders
def collapse_equation_fragments(text):
    if not isinstance(text, str):
        return text

    equation_fragment_pattern = re.compile(
        r"\[\[EQUATION\]\]"
        r"[^.!?\n]{0,2000}"
        r"\[\[EQUATION\]\]"
        r"[^.!?\n]{0,2000}",
        flags=re.IGNORECASE
    )

    def replace_equation_fragment(match):
        segment = match.group(0)

        math_indicators = 0
        math_indicators += segment.count("=")
        math_indicators += segment.count("+")
        math_indicators += segment.count("^")
        math_indicators += segment.count("_")
        math_indicators += segment.count("{")
        math_indicators += segment.count("}")

        if math_indicators >= 2:
            return " [[EQUATION]] "

        return segment

    text = equation_fragment_pattern.sub(
        replace_equation_fragment,
        text
    )

    # Catch cases such as [[EQUATION]] = [[EQUATION]] = [[EQUATION]]
    text = re.sub(
        r"\[\[EQUATION\]\]"
        r"(?:\s*[_^]\w+)?"
        r"(?:\s*[+=-]\s*"
        r"\[\[EQUATION\]\]"
        r"(?:\s*[_^]\w+)?)"
        r"+",
        " [[EQUATION]] ",
        text
    )

    return text


# Remove unwanted standalone symbols
def remove_symbol_noise(text):
    if not isinstance(text, str):
        return text

    text = re.sub(
        r"/\)",
        " ",
        text
    )

    text = re.sub(
        r"\(/",
        " ",
        text
    )

    text = re.sub(
        r"\(\)",
        " ",
        text
    )

    # Remove all asterisks
    text = text.replace(
        "*",
        " "
    )

    # Remove standalone slash
    text = re.sub(
        r"(?<!\w)/(?!\w)",
        " ",
        text
    )

    # Remove standalone parentheses
    text = text.replace(
        "(",
        " "
    )

    text = text.replace(
        ")",
        " "
    )

    return text


# Final pass to normalize the cleaned text
def final_text_cleanup(text):
    if not isinstance(text, str):
        return text

    text = normalize_placeholders(text)
    text = collapse_equation_fragments(text)
    text = clean_placeholder_parentheses(text)
    text = collapse_consecutive_placeholders(text)
    text = remove_symbol_noise(text)
    text = normalize_placeholders(text)
    text = collapse_equation_fragments(text)
    text = re.sub(
        r"\\+",
        " ",
        text
    )

    text = text.replace(
        "~",
        " "
    )

    text = re.sub(
        r"[{}]",
        " ",
        text
    )

    text = re.sub(
        r"(?<!\w)[_^]+(?!\w)",
        " ",
        text
    )

    # Convert [[EQUATION]] to the required [EQUATION] format
    text = re.sub(
        r"\[\[EQUATION\]\]",
        "[EQUATION]",
        text
    )

    # Make sure consecutive equation tags become one
    text = re.sub(
        r"(?:\s*\[EQUATION\]\s*){2,}",
        " [EQUATION] ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# Complete cleaning pipeline
def clean_mgtbench_pipeline(text):
    if not isinstance(text, str):
        return text

    text = text.replace(
        "\r",
        " "
    )

    text = text.replace(
        "\n",
        " "
    )

    text = text.replace(
        "\\n",
        " "
    )

    text = text.replace(
        "\\t",
        " "
    )

    text = clean_code_texts(text)
    text = clean_url(text)
    text = clean_foreign_script(text)
    text = clean_complexity_notation(text)
    text = strip_reference_list(text)
    text = clean_citations(text)
    text = clean_list_numbering(text)
    text = clean_math_texts(text)
    text = clean_residual_math_noise(text)
    text = final_text_cleanup(text)

    return text


# Clean the MGTBench AI dataset
def clean_mgtbench_ai_dataset(dataset):
    dataset = dataset.copy()
    original_rows = len(dataset)
    dataset["_before_being_cleaned"] = dataset["text"]
    dataset = dataset.dropna(
        subset=["text"]
    )
    dataset["text"] = (
        dataset["text"]
        .astype(str)
        .str.strip()
    )

    dataset = dataset[
        dataset["text"] != ""
    ]

    dataset["text"] = (
        dataset["text"]
        .apply(clean_mgtbench_pipeline)
    )

    dataset["text"] = (
        dataset["text"]
        .str.strip()
    )

    dataset = dataset[
        dataset["text"] != ""
    ]

    # Remove rows containing only placeholders
    only_placeholder = dataset["text"].str.fullmatch(
        rf"(?:{PLACEHOLDER_PATTERN}\s*)+"
    )

    only_equation = dataset["text"].str.fullmatch(
        r"\s*\[EQUATION\]\s*"
    )

    dataset = dataset[
        ~only_placeholder
    ]

    dataset = dataset[
        ~only_equation
    ]

    dataset = dataset.reset_index(
        drop=True
    )

    before_after = pd.DataFrame({
        "before_being_cleaned":
            dataset["_before_being_cleaned"],

        "after_text":
            dataset["text"]
    })

    before_after = (
        before_after
        .reset_index(drop=True)
    )

    dataset = dataset.drop(
        columns=["_before_being_cleaned"]
    )

    cleaning_stats = {
        "original_rows":
            original_rows,

        "cleaned_rows":
            len(dataset),

        "rows_removed":
            original_rows - len(dataset),

        "missing_values":
            dataset.isna()
            .sum()
            .to_dict(),

        "empty_texts":
            dataset["text"]
            .str.strip()
            .eq("")
            .sum(),

        "exact_duplicate_rows":
            dataset.duplicated()
            .sum(),

        "unique_files":
            dataset["file"].nunique()
            if "file" in dataset.columns
            else None
    }

    return (
        dataset,
        before_after,
        cleaning_stats
    )


# Run the cleaning using your existing mgtbench_ai dataset
mgtbench_ai_cleaned, mgtbench_before_after, cleaning_stats = (
    clean_mgtbench_ai_dataset(
        mgtbench_ai
    )
)
print("MGTBENCH AI CLEANING RESULTS")

print(
    f"\nOriginal rows: "
    f"{cleaning_stats['original_rows']:,}"
)

print(
    f"Cleaned rows:  "
    f"{cleaning_stats['cleaned_rows']:,}"
)

print(
    f"Rows removed:  "
    f"{cleaning_stats['rows_removed']:,}"
)

print(
    "\nEmpty texts after cleaning:"
)

print(
    cleaning_stats["empty_texts"]
)

print(
    "\nExact duplicate rows:"
)

print(
    cleaning_stats["exact_duplicate_rows"]
)

print(
    "\nUnique source files:"
)

print(
    cleaning_stats["unique_files"]
)


# Display before and after examples
print("\nBEFORE vs AFTER CLEANING")

ipd.display(
    mgtbench_before_after.head(20)
)


# Display cleaned dataset
print("\nCLEANED DATASET SAMPLE")

ipd.display(
    mgtbench_ai_cleaned.head(10)
)


# Check whether unwanted noise remains
print("\nFINAL NOISE CHECK")


remaining_backslashes = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\\",
        regex=True,
        na=False
    )
    .sum()
)

remaining_tildes = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        "~",
        regex=False,
        na=False
    )
    .sum()
)

consecutive_equations = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\[EQUATION\]\s+\[EQUATION\]",
        regex=True,
        na=False
    )
    .sum()
)

remaining_asterisks = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"\*",
        regex=True,
        na=False
    )
    .sum()
)

remaining_slash_parentheses = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"/\)",
        regex=True,
        na=False
    )
    .sum()
)

remaining_parentheses = (
    mgtbench_ai_cleaned["text"]
    .str.contains(
        r"[\(\)]",
        regex=True,
        na=False
    )
    .sum()
)


print(
    f"Texts containing '\\\\': "
    f"{remaining_backslashes}"
)

print(
    f"Texts containing '~': "
    f"{remaining_tildes}"
)

print(
    f"Texts with consecutive [EQUATION]: "
    f"{consecutive_equations}"
)

print(
    f"Texts containing '*': "
    f"{remaining_asterisks}"
)

print(
    f"Texts containing '/)': "
    f"{remaining_slash_parentheses}"
)

print(
    f"Texts containing parentheses: "
    f"{remaining_parentheses}"
)


# Show random before and after examples
print("\nRANDOM BEFORE vs AFTER EXAMPLES")


sample_size = min(
    10,
    len(mgtbench_ai_cleaned)
)


if sample_size > 0:

    sample_after = (
        mgtbench_ai_cleaned
        .sample(
            n=sample_size,
            random_state=42
        )
    )

    original_lookup = (
        mgtbench_ai
        .drop_duplicates("id")
        .set_index("id")["text"]
    )

    for i, (_, row) in enumerate(
        sample_after.iterrows(),
        start=1
    ):

        original_id = row["id"]

        if original_id in original_lookup.index:
            original_text = (
                original_lookup.loc[
                    original_id
                ]
            )
        else:
            original_text = (
                "[Original text not found]"
            )

        print(
            f"\nEXAMPLE {i}"
        )

        print(
            f"ID: {original_id}"
        )

        print(
            f"FILE: {row['file']}"
        )

        print(
            "\nBEFORE BEING CLEANED:"
        )

        print(
            original_text
        )

        print(
            "\nAFTER CLEANING:"
        )

        print(
            row["text"]
        )

        print(
            "\nCHANGED:"
        )

        print(
            "YES"
            if str(original_text)
            != str(row["text"])
            else "NO"
        )


# Find every row where the text changed
comparison = (
    mgtbench_ai_cleaned
    .copy()
)

comparison["before_being_cleaned"] = (
    comparison["id"]
    .map(
        mgtbench_ai
        .drop_duplicates("id")
        .set_index("id")["text"]
    )
)

comparison["after_text"] = (
    comparison["text"]
)

comparison["text_changed"] = (
    comparison["before_being_cleaned"]
    != comparison["after_text"]
)

changed_rows = comparison[
    comparison["text_changed"]
].copy()


print("\nCHANGED TEXTS")

print(
    f"\nRows with text changes: "
    f"{len(changed_rows):,}"
)

ipd.display(
    changed_rows[
        [
            "id",
            "file",
            "before_being_cleaned",
            "after_text"
        ]
    ].head(20)
)


# Prepare the before/after output
before_after_output = changed_rows[
    [
        "id",
        "file",
        "before_being_cleaned",
        "after_text"
    ]
].copy()

print(
    "\nBefore/after comparison is ready."
)

ipd.display(
    before_after_output.head(10)
)

MGTBENCH AI CLEANING RESULTS

Original rows: 313,775
Cleaned rows:  313,619
Rows removed:  156

Empty texts after cleaning:
0

Exact duplicate rows:
0

Unique source files:
46

BEFORE vs AFTER CLEANING


,before_being_cleaned,after_text
0,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,The results presented in this study shed light...,The results presented in this study shed light...
2,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...



CLEANED DATASET SAMPLE


,id,text,file
0,0,"In this report, we present and compare two met...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
1,2,The results presented in this study shed light...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
2,4,Kr sensitivities and uncertainties In this sec...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
3,5,"In this section, we aim to quantify the non-Ga...",../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
4,6,The key innovation of this study is the replac...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
5,7,[EQUATION] Our research emphasizes the importa...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
6,8,Motion of stars relative to their local inters...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
7,9,IceCube studies a diverse range of physics top...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
8,10,The observed spectral energy distribution SED ...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...
9,11,We established the photometric catalogs in eac...,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...



FINAL NOISE CHECK
Texts containing '\\': 0
Texts containing '~': 0
Texts with consecutive [EQUATION]: 0
Texts containing '*': 0
Texts containing '/)': 0
Texts containing parentheses: 0

RANDOM BEFORE vs AFTER EXAMPLES

EXAMPLE 1
ID: 1352
FILE: Electrical_engineering_wiki_new.json

BEFORE BEING CLEANED:
In the early universe, Eqs.~\\eqref{dq} are slightly modified. While the equation of motion for $\\mathbf{D}$ remains unchanged, the second line becomes $\\dot{\\mathbf{Q}} = \\mu \\mathbf{D} \\times \\mathbf{Q} - 4 H \\left( \\omega / \\mu \\right) \\mathbf{B}$, where $H$ represents the Hubble constant. Despite the time-dependence induced by the universe's expansion, certain quantities remain strictly conserved. Specifically, $\\mathbf{B} \\cdot \\mathbf{D}$ maintains its value, which can be interpreted as the angular momentum along the gravitational field. Additionally, $\\mathbf{D} \\cdot \\mathbf{Q} + \\frac{\\omega}{\\mu} \\mathbf{B} \\cdot \\mathbf{D}$ remains constant and upholds

,id,file,before_being_cleaned,after_text
0,0,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,2,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The results presented in this study shed light...,The results presented in this study shed light...
2,4,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,5,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,6,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,7,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,8,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,9,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,10,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,11,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...



Before/after comparison is ready.


,id,file,before_being_cleaned,after_text
0,0,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this report, we present and compare two met...","In this report, we present and compare two met..."
1,2,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The results presented in this study shed light...,The results presented in this study shed light...
2,4,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Kr sensitivities and uncertainties\n\nIn this ...,Kr sensitivities and uncertainties In this sec...
3,5,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,"In this section, we aim to quantify the non-Ga...","In this section, we aim to quantify the non-Ga..."
4,6,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The key innovation of this study is the replac...,The key innovation of this study is the replac...
5,7,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,\\section{Results and Discussion}\n\nOur resea...,[EQUATION] Our research emphasizes the importa...
6,8,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,Motion of stars relative to their local inters...,Motion of stars relative to their local inters...
7,9,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,IceCube studies a diverse range of physics top...,IceCube studies a diverse range of physics top...
8,10,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,The observed spectral energy distribution (SED...,The observed spectral energy distribution SED ...
9,11,../AIGen-TASK3/gpt35/Physics_task3_resultsgpt-...,We established the photometric catalogs in eac...,We established the photometric catalogs in eac...


In [32]:
def clean_bawe_corpus_dataset(dataset):

    pass